In [ ]:
PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "standardized_exports"

RESULTS_DIR = PROJECT_ROOT / "results"

FORECAST_DIR = PROJECT_ROOT / "forecasts"

FIGURE_DIR = PROJECT_ROOT / "figures"

for p in [
    RESULTS_DIR,
    FORECAST_DIR,
    FIGURE_DIR
]:
    p.mkdir(exist_ok=True)

In [ ]:
CSV_FILES = sorted(
    DATA_DIR.glob("*.csv")
)

DATASET_NAMES = [
    f.stem
    for f in CSV_FILES
]

print(
    f"{len(DATASET_NAMES)} datasets found"
)

In [ ]:
def load_dataset(file_path):

    df = pd.read_csv(
        file_path,
        low_memory=False
    )

    df["ds"] = pd.to_datetime(
        df["ds"]
    )

    return df

In [ ]:
def dataset_summary(df):

    return {
        "n_rows": len(df),
        "n_series": df["unique_id"].nunique(),
        "start": df["ds"].min(),
        "end": df["ds"].max()
    }

In [ ]:
HORIZON_MAP = {

    "agri": 12,

    "energy": 24,

    "favorita-train": 28,
    "favorita-transaction": 28,

    "m4-daily": 14,
    "m4-hourly": 48,
    "m4-weekly": 13,
    "m4-monthly": 18,
    "m4-quarterly": 8,
    "m4-yearly": 6,

    "traffic": 24
}

In [ ]:
def get_horizon(dataset_name):

    for suffix in [
        "_small",
        "_medium",
        "_large"
    ]:

        if dataset_name.endswith(suffix):

            dataset_name = dataset_name.replace(
                suffix,
                ""
            )

            break

    return HORIZON_MAP[dataset_name]

In [ ]:
for file in CSV_FILES:
    dataset_name = file.stem

    horizon = get_horizon(
        dataset_name
    )

In [ ]:
def train_test_split_series(
    df,
    horizon
):

    train_parts = []
    test_parts = []

    for _, group in df.groupby(
        "unique_id"
    ):

        group = group.sort_values(
            "ds"
        )

        if len(group) <= horizon:
            continue

        train_parts.append(
            group.iloc[:-horizon]
        )

        test_parts.append(
            group.iloc[-horizon:]
        )

    train_df = pd.concat(
        train_parts,
        ignore_index=True
    )

    test_df = pd.concat(
        test_parts,
        ignore_index=True
    )

    return train_df, test_df


In [ ]:
def smape(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    denominator = (
        np.abs(y_true)
        + np.abs(y_pred)
    )

    denominator = np.where(
        denominator == 0,
        1,
        denominator
    )

    return (
        np.mean(
            2 *
            np.abs(y_true - y_pred)
            / denominator
        )
        * 100
    )

In [ ]:
def calculate_metrics(
    y_true,
    y_pred
):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    return {

        "MAE":
            mean_absolute_error(
                y_true,
                y_pred
            ),

        "RMSE":
            np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred
                )
            ),

        "SMAPE":
            smape(
                y_true,
                y_pred
            )
    }

In [ ]:
def get_covariates(
    dataset_name,
    train_df,
    test_df
):

    if "favorita" not in dataset_name.lower():

        return None, None

    cols = [

        c

        for c in train_df.columns

        if c not in [
            "unique_id",
            "ds",
            "y"
        ]
    ]

    if not cols:
        return None, None

    return (
        train_df[cols],
        test_df[cols]
    )

In [ ]:
def save_forecasts(
    forecast_df,
    dataset_name,
    model_name
):

    forecast_df.to_csv(

        FORECAST_DIR
        /
        f"{dataset_name}_{model_name}.csv",

        index=False
    )

In [ ]:
def build_result_row(
    dataset_name,
    model_name,
    horizon,
    runtime,
    metrics,
    df
):

    return {

        "dataset":
            dataset_name,

        "model":
            model_name,

        "horizon":
            horizon,

        "runtime_seconds":
            runtime,

        "n_rows":
            len(df),

        "n_series":
            df["unique_id"].nunique(),

        **metrics
    }

In [ ]:
def plot_forecast(
    actual,
    forecast,
    title=""
):

    plt.figure(
        figsize=(12, 5)
    )

    plt.plot(
        actual,
        label="Actual"
    )

    plt.plot(
        forecast,
        label="Forecast"
    )

    plt.title(title)

    plt.legend()

    plt.show()

In [ ]:
print("Setup Complete")

print(
    f"Datasets Found: {len(DATASET_NAMES)}"
)

for ds in DATASET_NAMES:
    print(ds)